The purpose of this notebook is to demonstrate how to perform streamflow evaluation using NOAA National Water Model retrospective predictions.

In [ ]:
import s3fs
import pandas as pd
import xarray as xr
import geopandas as gpd
from datetime import datetime
from dask.distributed import Client

import hf_hydrodata as hf



import hydrodata_creds # local file containing my hydrodata credentials

In [ ]:
# You need to register on https://hydrogen.princeton.edu/pin 
# and run the following with your registered information
# before you can use the hydrodata utilities
hf.register_api_pin(hydrodata_creds.username,
                    hydrodata_creds.pin)

In [ ]:
# Define the time range of our analysis
start_date = '2020-01-01'
end_date = '2022-01-01'

Gather streamflow observations from HydroData.

In [ ]:
obs_sites = hf.get_site_variables(datase='usgs_nwis',
                                  variable='streamflow',
                                  date_start=start_date,
                                  date_end=end_date,
                                  huc_id=['02030105'],
                                  grid = 'conus2',
                                  )  

In [ ]:
obs_sites.head()

Filter our data to include only "hourly" observations to align with the resolution of NWM predictions.

In [ ]:
obs_sites = obs_sites[obs_sites.temporal_resolution == 'daily']

Identify all of the gauges that exist in our region of interest.

In [ ]:
usgs_gauges = obs_sites.site_id.unique()
usgs_gauges

Collect NWM retrospective v3.0 streamflow data for each of these gauges.

In [ ]:
# Path to NWM channel routing output
s3_url = 's3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/chrtout.zarr'

Connect to the public store of NWM data. This is a very large dataset and as a result, it's stored using the Zarr format to enable lazy loading and buffered data reading.

In [ ]:

%%time 
ds = xr.open_zarr(
    store=s3_url,
    consolidated=True,
    storage_options={
        "anon": True,
        "client_kwargs": {"region_name": "us-east-1"}
    }
)
ds

Select data for the gauges identified above. The command below will give us an array of True and False, where "True" corresponds with the `usgs_gauges` identified above and "False" everywhere else. First we need to format the values in our `usgs_gauges` list into the format used in the Zarr store, i.e. 15 character strings.

In [ ]:
formatted_usgs_gauges = [f"{gauge_id:>15}".encode('ascii') for gauge_id in usgs_gauges]
print(f'The first gauge now looks like this: {formatted_usgs_gauges[0]}')

In [ ]:
mask = ds.gage_id.isin(formatted_usgs_gauges)

Use this mask to isolate the data within the Zarr store for the gauge locations.

In [ ]:
res = ds.where(mask.compute(), drop=True)
res

To use these data in the evaluation framework we need to put them into the expected data format. The evaluation library is expecting a `pandas.DataFrame` structured in the way:

|date|Series 1| Series 2| ... | Series N|
|---|---|---|---|---|
|2018-01-01| 0.0 | 0.3 | ... | 1.1 |
|2018-01-02| 0.1 | 0.2 | ... | 1.0 |
|...| ... | ... | ... | ... |
|2020-01-01| 0.3 | 0.0 | ... | 1.3 |

The first step is to get the modeled NWM results from the Dask `DataSet` above into a Pandas `DataFrame`.

In [ ]:
start_date = datetime(2020,1,1)
end_date = datetime(2022,1,1)

In [ ]:
nwm_df = res.drop_vars(['elevation', 'latitude', 'longitude','order', 'feature_id']) \
           .sel(time=slice(start_date, end_date)) \
           .streamflow.to_dataframe() \
           .reset_index()

In [ ]:
# format gage_ids from bytestrings to str objects
nwm_df.gage_id = nwm_df.gage_id.str.decode('utf-8').str.strip()

# add "nwm" suffix to gage_id column. This is so that our
# column names are more descriptive after we pivot the data.
nwm_df.gage_id = nwm_df.gage_id + '_nwm'

nwm_df

In [ ]:
# rename time column to 'date' to be consistent with the format 
# expected by the evaluation framework
nwm_df.rename(columns={'time': 'date'}, inplace=True) 

# pivot the data into the form expected by the evaluation framework
nwm_df = pd.pivot(nwm_df, values='streamflow', index='date', columns='gage_id') 

# remove the 'name' label on the columns axis to make things cleaner.
nwm_df = nwm_df.rename_axis(None, axis=1)

# reset the index
nwm_df.reset_index(inplace=True)

nwm_df

Get observation for each of the sites we identified at the beginning of the notebook. These should still be stored in the `usgs_gauges` variable.

In [ ]:
usgs_gauges

In [ ]:
# use get_point_data to retrieve observation data for each of the usgs_gauge ids
obs_df = hf.get_point_data(dataset="usgs_nwis",
                            variable="streamflow",
                            temporal_resolution="hourly",
                            aggregation='mean',
                            date_start=start_date,
                            date_end=end_date,
                            site_ids= list(usgs_gauges))

In [ ]:
# add the 'obs' suffix to the observation columns so that these
# can easily be distinguished from the 'nwm' data 

obs_df = obs_df.set_index('date') \
            .add_suffix('_obs') \
            .reset_index()

Convert the date column to a datetime[64]

In [ ]:
obs_df.date = pd.to_datetime(obs_df.date)

In [ ]:
obs_df.head()

Combine the NWM and Observation DataFrames

In [ ]:
df = nwm_df.merge(obs_df, left_on='date', right_on='date', how='inner')

In [ ]:
df.head()

Perform evaluation on each site.

In [ ]:
# Import the Evaluation library from the project root.
import numpy as np
import sys
from pathlib import Path

sys.path.append(str((Path.cwd().absolute() / "../../../src").resolve()))
from cssi_evaluation import evaluation_metrics
from cssi_evaluation.model_evaluation import METRICS_DICT
METRICS_DICT.pop('condon') # remove condon because this take a different input

In [ ]:
results = {}
for gauge in usgs_gauges:
    nwm_col = f'{gauge}_nwm'
    obs_col = f'{gauge}_obs'

    # subset the data we're interested in and ignore NaNs
    dat = df[[nwm_col, obs_col]].dropna()

    results[gauge] = {}
    for metric_name, metric_func in METRICS_DICT.items():
        results[gauge][metric_name] =metric_func(np.array(dat[nwm_col].to_list()),
                                                 np.array(dat[obs_col].to_list()))

In [ ]:
res = pd.DataFrame(results).T
res